# 🏇 JRA 全レース取得 (2020-2026)
以下の設定変数を変更して実行してください。指定した期間のデータを取得し、`SAVE_DIR` に保存します。

In [ ]:
# Google Driveをマウントする場合のみ実行してください
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ========================================
# 設定（ここを変更してください）
# ========================================
YEAR = 2024          # 対象年度 (例: 2024)
START_MONTH = 1      # 開始月 (1-12)
END_MONTH = 12       # 終了月 (1-12)
SAVE_DIR = '/content/drive/MyDrive/dai-keiba/data/raw' # 保存先フォルダ

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import io
import re
from datetime import datetime
import urllib.parse
import time
import random
from tqdm.auto import tqdm

def scrape_jra_race(url, existing_race_ids=None, max_retries=3):
    """
    Scrapes a single race page from JRA website with retry logic.
    Returns a pandas DataFrame matching the schema of database.csv.
    If existing_race_ids is provided and the race ID is found, returns None (skip).
    """
    print(f"Accessing URL: {url}...")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(url, headers=headers, timeout=15)
            response.encoding = 'EUC-JP'

            if response.status_code != 200:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt
                    print(f"Status {response.status_code}, retrying in {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                else:
                    print(f"Error: Status code {response.status_code}")
                    return None

            soup = BeautifulSoup(response.text, 'html.parser')

            # --- Metadata Extraction ---
            h1_elem = soup.select_one("div.header_line h1 .txt")
            full_text = h1_elem.text.strip() if h1_elem else ""
            if not full_text and soup.h1:
                full_text = soup.h1.text.strip()

            date_text = ""
            venue_text = ""
            race_num_text = ""
            kai = "01"
            day = "01"

            # Extract Date
            match_date = re.search(r'(\d{4}年\d{1,2}月\d{1,2}日)', full_text)
            if match_date:
                date_text = match_date.group(1)

            # Extract Venue, Kai, Day
            venues_str = "札幌|函館|福島|新潟|東京|中山|中京|京都|阪神|小倉"
            match_meta = re.search(rf'(\d+)回({venues_str})(\d+)日', full_text)
            if match_meta:
                kai = f"{int(match_meta.group(1)):02}"
                venue_text = match_meta.group(2)
                day = f"{int(match_meta.group(3)):02}"

            # Extract Race Num
            match_race = re.search(r'(\d+)レース', full_text)
            if match_race:
                r_val = int(match_race.group(1))
                race_num_text = f"{r_val}R"
                r_num = f"{r_val:02}"
            else:
                race_num_text = "10R"
                r_num = "10"

            # Race Name
            race_name_text = ""
            name_elem = soup.select_one(".race_name")
            if name_elem:
                race_name_text = name_elem.text.strip()

            # Grade
            grade_text = ""
            if "G1" in str(soup) or "ＧⅠ" in str(soup): grade_text = "G1"
            elif "G2" in str(soup) or "ＧⅡ" in str(soup): grade_text = "G2"
            elif "G3" in str(soup) or "ＧⅢ" in str(soup): grade_text = "G3"

            # --- Course, Distance, Weather, Condition ---
            header_text = soup.select_one("div.header_line").text if soup.select_one("div.header_line") else soup.text

            dist_type_match = re.search(r'(芝|ダ|ダート|障害)[^0-9]*(\d+)', header_text)
            course_type = ""
            distance = ""

            if dist_type_match:
                c_val = dist_type_match.group(1)
                d_val = dist_type_match.group(2)

                if "芝" in c_val: course_type = "芝"
                elif "ダ" in c_val: course_type = "ダート"
                elif "障" in c_val: course_type = "障害"

                distance = int(d_val)

            # Rotation
            rotation = ""
            rot_match = re.search(r'[（\(](右|左|直線)[）\)]', header_text)
            if rot_match:
                rotation = rot_match.group(1)
            else:
                 if "左" in header_text: rotation = "左"
                 elif "右" in header_text: rotation = "右"
                 elif "直線" in header_text: rotation = "直線"

            # Weather
            weather = ""
            w_match = re.search(r'天候\s*[:：]\s*(\S+)', soup.text)
            if w_match:
                weather = w_match.group(1).strip()

            # Condition
            condition = ""
            if course_type == "芝":
                 c_match = re.search(r'芝\s*[:：]\s*(\S+)', soup.text)
                 if c_match: condition = c_match.group(1).strip()
            elif course_type == "ダート":
                 c_match = re.search(r'ダート\s*[:：]\s*(\S+)', soup.text)
                 if c_match: condition = c_match.group(1).strip()

            if not condition:
                 c_match_gen = re.search(r'(?:芝|ダート)\s*[:：]\s*(\S+)', soup.text)
                 if c_match_gen: condition = c_match_gen.group(1).strip()

            # --- Table Extraction ---
            tables = soup.find_all('table')
            target_table = None
            for tbl in tables:
                if "着順" in tbl.text and "馬名" in tbl.text:
                    target_table = tbl
                    break

            if not target_table:
                print(f"Warning: Result table not found in {url}")
                return None

            rows = target_table.find_all('tr')
            data = []

            for row in rows:
                if row.find('th'):
                    continue

                cells = row.find_all('td')
                if not cells:
                    continue

                def get_text(idx):
                    if idx < len(cells):
                        return cells[idx].get_text(strip=True)
                    return ""

                # Extract Waku
                waku_text = ""
                if len(cells) > 1:
                    img = cells[1].find('img')
                    if img and 'alt' in img.attrs:
                        alt = img['alt']
                        m = re.search(r'枠(\d+)', alt)
                        if m:
                            waku_text = m.group(1)
                        else:
                            waku_text = alt

                # Extract Horse ID
                horse_id = ""
                if len(cells) > 3:
                    a_tag = cells[3].find('a')
                    if a_tag and 'href' in a_tag.attrs:
                        href = a_tag['href']
                        m = re.search(r'/horse/(\d+)', href)
                        if m:
                            horse_id = m.group(1)

                row_data = {
                    '着 順': get_text(0),
                    '枠': waku_text,
                    '馬 番': get_text(2),
                    '馬名': get_text(3),
                    'horse_id': horse_id,
                    '性齢': get_text(4),
                    '斤量': get_text(5),
                    '騎手': get_text(6),
                    'タイム': get_text(7),
                    '着差': get_text(8),
                    'コーナー 通過順': get_text(9),
                    '後3F': get_text(10),
                    '馬体重 (増減)': get_text(11),
                    '厩舎': get_text(12),
                    '人 気': get_text(13),
                    '単勝 オッズ': "0.0"
                }
                data.append(row_data)

            df = pd.DataFrame(data)

            # Add Metadata
            df['日付'] = date_text
            df['会場'] = venue_text
            df['レース番号'] = race_num_text
            df['レース名'] = race_name_text
            df['重賞'] = grade_text
            df['距離'] = distance
            df['コースタイプ'] = course_type
            df['天候'] = weather
            df['馬場状態'] = condition
            df['回り'] = rotation

            # ID Generation
            place_map = {
                "札幌": "01", "函館": "02", "福島": "03", "新潟": "04", "東京": "05",
                "中山": "06", "中京": "07", "京都": "08", "阪神": "09", "小倉": "10"
            }
            p_code = place_map.get(venue_text, "00")

            year = "2025"
            if date_text:
                year = date_text[:4]

            generated_id = f"{year}{p_code}{kai}{day}{r_num}"

            # SKIP CHECK
            if existing_race_ids and generated_id in existing_race_ids:
                print(f"Skipping {generated_id} (Already exists)")
                return None

            df['race_id'] = generated_id

            # Cleanups
            if '単勝 オッズ' in df.columns:
                df['単勝 オッズ'] = pd.to_numeric(df['単勝 オッズ'], errors='coerce').fillna(0.0)

            standard_columns = [
                "日付","会場","レース番号","レース名","重賞","着 順","枠","馬 番","馬名","性齢","斤量","騎手",
                "タイム","着差","人 気","単勝 オッズ","後3F","コーナー 通過順","厩舎","馬体重 (増減)","race_id",
                "距離","コースタイプ","天候","馬場状態","回り"
            ]

            for col in standard_columns:
                if col not in df.columns:
                    df[col] = ""

            df = df[standard_columns]

            print(f"✅ Scraped {len(df)} rows.")
            return df

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"Error: {e}, retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"❌ Failed after {max_retries} attempts: {e}")
                return None

    return None


# Parameter Map for Monthly Results
JRA_MONTH_PARAMS = {
    "2026": { "01": "E4", "02": "B2", "03": "80", "04": "4E", "05": "1C", "06": "EA", "07": "B8", "08": "86", "09": "54", "10": "22", "11": "F0", "12": "BE" },
    "2025": { "01": "3F", "02": "0D", "03": "DB", "04": "A9", "05": "77", "06": "45", "07": "13", "08": "E1", "09": "AF", "10": "1E", "11": "EC", "12": "D3" },
    "2024": { "01": "B3", "02": "81", "03": "4F", "04": "1D", "05": "EB", "06": "B9", "07": "87", "08": "55", "09": "23", "10": "92", "11": "60", "12": "2E" },
    "2023": { "01": "27", "02": "F5", "03": "C3", "04": "91", "05": "5F", "06": "2D", "07": "FB", "08": "C9", "09": "97", "10": "06", "11": "D4", "12": "A2" },
    "2022": { "01": "9B", "02": "69", "03": "37", "04": "05", "05": "D3", "06": "A1", "07": "6F", "08": "3D", "09": "0B", "10": "7A", "11": "48", "12": "16" },
    "2021": { "01": "0F", "02": "DD", "03": "AB", "04": "79", "05": "47", "06": "15", "07": "E3", "08": "B1", "09": "7F", "10": "EE", "11": "BC", "12": "8A" },
    "2020": { "01": "83", "02": "51", "03": "1F", "04": "ED", "05": "BB", "06": "89", "07": "57", "08": "25", "09": "F3", "10": "62", "11": "30", "12": "FE" }
}

def scrape_jra_year(year_str, start_date=None, end_date=None, save_callback=None, existing_race_ids=None):
    """
    Scrapes races for a given year and date range with progress tracking.
    """

    if year_str not in JRA_MONTH_PARAMS:
        print(f"Year {year_str} not supported in parameter map.")
        return

    params = JRA_MONTH_PARAMS[year_str]
    base_url = "https://www.jra.go.jp/JRADB/accessS.html"

    # Determine months to iterate
    start_m = 1
    end_m = 12

    if start_date:
        start_m = start_date.month
    if end_date:
        end_m = end_date.month

    # Cap at Today
    from datetime import date
    today = date.today()

    if end_date:
        actual_end_date = min(end_date, today)
    else:
        actual_end_date = today

    print(f"=== Starting JRA Bulk Scraping for {year_str} ===")
    print(f"Period: {start_date or 'Start'} - {actual_end_date}")
    print(f"Using random delays (1.0-2.0s) to avoid rate limiting")

    if int(year_str) == today.year:
        end_m = min(end_m, today.month)
    elif int(year_str) > today.year:
        print(f"Year {year_str} is in the future. Stopping.")
        return

    failed_races = []
    total_processed = 0

    for m in range(start_m, end_m + 1):
        month = f"{m:02}"
        if month not in params:
            continue

        suffix = params[month]
        try:
            ym = int(year_str + month)
            prefix = "pw01skl00" if ym >= 202512 else "pw01skl10"
        except:
            prefix = "pw01skl10"

        cname = f"{prefix}{year_str}{month}/{suffix}"

        print(f"\n📅 Fetching {year_str}/{month}...")

        try:
            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
            }
            response = requests.post(base_url, data={"cname": cname}, headers=headers, timeout=15)
            response.encoding = 'cp932'

            if response.status_code != 200:
                print(f"❌ Failed to fetch {cname} (Status {response.status_code})")
                continue

            soup = BeautifulSoup(response.text, 'html.parser')

            race_cnames = []
            links = soup.find_all('a')
            for link in links:
                onclick = link.get('onclick', '')
                match = re.search(r"doAction\('[^']+',\s*'([^']+)'\)", onclick)
                if match:
                    c = match.group(1)
                    if c.startswith('pw01srl'):
                        race_cnames.append(c)

            race_cnames = sorted(list(set(race_cnames)))
            print(f"  Found {len(race_cnames)} race days")

            for day_cname in tqdm(race_cnames, desc=f"  {year_str}/{month}", leave=False):
                resp_day = requests.post(base_url, data={"cname": day_cname}, headers=headers, timeout=15)
                resp_day.encoding = 'cp932'
                soup_day = BeautifulSoup(resp_day.text, 'html.parser')

                d_h1 = soup_day.select_one("div.header_line h1 .txt")
                full_d_text = d_h1.text.strip() if d_h1 else (soup_day.h1.text.strip() if soup_day.h1 else "")

                current_day_date = None
                kai_str = "01"
                day_str = "01"
                venue_str = ""
                p_code = "00"

                match_day_date = re.search(r'(\d{4})年(\d{1,2})月(\d{1,2})日', full_d_text)
                if match_day_date:
                    y, mo, d_day = map(int, match_day_date.groups())
                    current_day_date = datetime(y, mo, d_day).date()

                    if start_date and current_day_date < start_date:
                        continue
                    if end_date and current_day_date > end_date:
                        continue

                venues_ptn = "札幌|函館|福島|新潟|東京|中山|中京|京都|阪神|小倉"
                match_meta = re.search(rf'(\d+)回({venues_ptn})(\d+)日', full_d_text)
                if match_meta:
                    kai_str = f"{int(match_meta.group(1)):02}"
                    venue_str = match_meta.group(2)
                    day_str = f"{int(match_meta.group(3)):02}"

                    place_map = {
                        "札幌": "01", "函館": "02", "福島": "03", "新潟": "04", "東京": "05",
                        "中山": "06", "中京": "07", "京都": "08", "阪神": "09", "小倉": "10"
                    }
                    p_code = place_map.get(venue_str, "00")

                race_list_items = []
                all_anchors = soup_day.find_all('a')
                for a in all_anchors:
                    onclick = a.get('onclick', '')
                    match_sde = re.search(r"doAction\s*\(\s*['\"][^'\"]+['\"]\s*,\s*['\"](pw01sde[^'\"]+)['\"]\s*\)", onclick)
                    href = a.get('href', '')

                    final_url = ""
                    if match_sde:
                        final_url = f"{base_url}?CNAME={match_sde.group(1)}"
                    elif 'pw01sde' in href:
                        if 'CNAME=' in href:
                             final_url = urllib.parse.urljoin(base_url, href)
                        else:
                             final_url = urllib.parse.urljoin(base_url, href)

                    if final_url:
                        txt = a.text.strip()
                        img = a.find('img')
                        if not txt and img and 'alt' in img.attrs:
                            txt = img['alt']

                        r_num = -1
                        r_num_match = re.search(r'(\d+)R', txt)
                        if r_num_match:
                             r_num = int(r_num_match.group(1))

                        race_list_items.append((final_url, r_num))

                seen_urls = set()
                unique_races = []
                for url, r_num in race_list_items:
                    if url not in seen_urls:
                        unique_races.append((url, r_num))
                        seen_urls.add(url)

                unique_races.sort(key=lambda x: x[1])

                for r_link, r_num in unique_races:
                    # Pre-check skip
                    if r_num != -1 and p_code != "00" and current_day_date:
                         y_str = str(y)
                         r_num_str = f"{r_num:02}"
                         generated_id = f"{y_str}{p_code}{kai_str}{day_str}{r_num_str}"

                         if existing_race_ids and generated_id in existing_race_ids:
                             continue

                    # Fetch with retry
                    df = scrape_jra_race(r_link, existing_race_ids=existing_race_ids)

                    if df is not None and not df.empty:
                        if save_callback:
                            save_callback(df)
                        total_processed += 1
                    else:
                        # Log failure
                        race_id = generated_id if r_num != -1 else r_link
                        failed_races.append(race_id)

                    # Rate limiting with random delay
                    time.sleep(random.uniform(1.0, 2.0))

                    # Session keepalive
                    if total_processed % 10 == 0 and total_processed > 0:
                        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ {total_processed}件処理完了")

        except Exception as e:
            print(f"❌ Error processing month {month}: {e}")

    # Final summary
    print(f"\n{'='*50}")
    print(f"✅ スクレイピング完了")
    print(f"総処理件数: {total_processed}件")
    print(f"失敗件数: {len(failed_races)}件")

    if failed_races:
        print(f"\n⚠️ 失敗したレース:")
        for race_id in failed_races[:10]:
            print(f"  - {race_id}")
        if len(failed_races) > 10:
            print(f"  ... 他 {len(failed_races) - 10}件")


In [ ]:
# 実行ブロック
import os
from datetime import date
import calendar

if YEAR:
    # Saveディレクトリの作成
    os.makedirs(SAVE_DIR, exist_ok=True)

    s_date = date(int(YEAR), int(START_MONTH), 1)
    last_day = calendar.monthrange(int(YEAR), int(END_MONTH))[1]
    e_date = date(int(YEAR), int(END_MONTH), last_day)

    # 未来の日付は検索しないように制限
    today = date.today()
    if e_date > today:
        e_date = today

    save_path = os.path.join(SAVE_DIR, 'database.csv')

    # 既存データの読み込みとチェック
    existing_race_ids = set()
    if os.path.exists(save_path):
        print('既存データを読み込み中...')
        try:
            existing_df = pd.read_csv(save_path, low_memory=False)
            if 'race_id' in existing_df.columns:
                # race_idを文字列に変換
                existing_df['race_id'] = existing_df['race_id'].astype(str).str.replace(r'\.0$', '', regex=True)

                # 必須カラムのリスト
                required_columns = ['馬名', 'horse_id', '距離', 'コースタイプ', '天候', '馬場状態', '日付', '会場']

                # すべての必須データが揃っている行のみを「完全」とみなす
                complete_mask = True
                for col in required_columns:
                    if col in existing_df.columns:
                        complete_mask = complete_mask & existing_df[col].notna() & (existing_df[col] != '') & (existing_df[col] != 'nan')

                complete_races = existing_df[complete_mask]['race_id'].unique()
                existing_race_ids = set(complete_races)

                total_races = existing_df['race_id'].nunique()
                complete_count = len(existing_race_ids)
                print(f'既存データ: {total_races}レース中 {complete_count}レースが完全')
                print(f'不完全または欠損データがある{total_races - complete_count}レースは再取得対象')
        except Exception as e:
            print(f'既存データの読み込みエラー（新規作成します）: {e}')

    print(f'{YEAR}年のデータを {s_date} から {e_date} まで取得します...')
    print(f'保存先: {save_path}')
    scrape_jra_year(str(YEAR), start_date=s_date, end_date=e_date, save_callback=lambda df: df.to_csv(save_path, mode='a', header=not os.path.exists(save_path), index=False), existing_race_ids=existing_race_ids)
    print('完了しました。')
else:
    print('年度が設定されていません。')

In [ ]:
# 実行ブロック
import os
import pandas as pd
from datetime import date
import calendar

if YEAR:
    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path = os.path.join(SAVE_DIR, 'database.csv')

    # --- 【追加】既存データの読み込みとID抽出 ---
    existing_race_ids = set()
    if os.path.exists(save_path):
        try:
            # race_id列のみを読み込んで高速化
            temp_df = pd.read_csv(save_path, usecols=['race_id'], dtype={'race_id': str})
            existing_race_ids = set(temp_df['race_id'].unique())
            print(f"既存データを確認: {len(existing_race_ids)} 件のレースをスキップ対象に設定しました。")
        except Exception as e:
            print(f"既存データの読み込み中にエラー（新規作成します）: {e}")
    # ------------------------------------------

    s_date = date(int(YEAR), int(START_MONTH), 1)
    last_day = calendar.monthrange(int(YEAR), int(END_MONTH))[1]
    e_date = date(int(YEAR), int(END_MONTH), last_day)

    today = date.today()
    if e_date > today:
        e_date = today

    print(f'{YEAR}年のデータを {s_date} から {e_date} まで取得します...')

    # scrape_jra_year に existing_race_ids を渡すように修正
    scrape_jra_year(
        str(YEAR),
        start_date=s_date,
        end_date=e_date,
        existing_race_ids=existing_race_ids, # ここで渡す
        save_callback=lambda df: df.to_csv(save_path, mode='a', header=not os.path.exists(save_path), index=False)
    )
    print('完了しました。')
else:
    print('年度が設定されていません。')